In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
import re

In [ ]:
from datetime import datetime
from isoweek import Week
from cmdstanpy import CmdStanModel
from cmdstanpy import from_csv
import glob

import matplotlib.cm as cm
col1 = plt.cm.inferno(0.3)
col2 = plt.cm.inferno(0.55)
col3 = plt.cm.inferno(0.8)

In [ ]:
dt_discrete = 1/8

In [ ]:
csv_files = glob.glob('stan_output/sinusoid_2025/*20250801092152*.csv')  # dt=1/8, better prior for S0, 3000 iterations, good diagnostics all around 
print(csv_files)

fit = from_csv(csv_files)

In [ ]:
for var in ['S0', 'logx_I0', 'beta0', 'dbeta', 'betaphase', 'sigma_obs', 'rho', 'delta']:
    print(f"{var}: {fit.stan_variable(var).mean()}")

In [ ]:
df_draws = fit.draws_pd()
lp = df_draws["lp__"]  # log posterior column
imax = np.argmax(lp)  # index of sample with highest lp__
best_draw = df_draws.iloc[imax]

In [ ]:
def best_draw_to_dict(row):
    """
    Convert a series of Stan draws into a structured dict,
    grouping vector parameters like param[1], param[2], ... into arrays.
    """
    result = {}
    for col in row.index:
        # Attempt to match something like "paramName[123]"
        m = re.match(r"(.*)\[(\d+)\]$", col)
        if m:
            base_name = m.group(1)
            idx = int(m.group(2))  # 1-based index from Stan
            val = row[col]

            if base_name not in result:
                # store them in a dict first, convert to array after.
                result[base_name] = {}
            result[base_name][idx] = val
        else:
            # it is a scalar param or a special column like chain__, iter__, ...
            result[col] = row[col]

    # Convert any dict-of-indices to a list or array
    # Stan uses 1-indexing
    for k, v in list(result.items()):
        if isinstance(v, dict):
            # v is a dictionary of indices -> values
            max_idx = max(v.keys())
            # create an array of length = max index
            arr = np.empty(max_idx, dtype=float)
            for i in range(1, max_idx + 1):
                arr[i - 1] = v[i]  # shift to zero-indexing
            result[k] = arr
    return result

best_draw_dict = best_draw_to_dict(best_draw)


In [ ]:
# We must translate rates into their continuous time versions: 
def DiscTimeRate(r, dt):
    return (1 - np.exp(-r*dt))/dt


mu = DiscTimeRate(1/(52 * 80.0), dt_discrete)

with_seasonality = True

beta0 = DiscTimeRate(best_draw_dict['beta0'], dt_discrete)
betaphase = best_draw_dict['betaphase']
gamma = DiscTimeRate(best_draw_dict['gamma'], dt_discrete)
delta = DiscTimeRate(best_draw_dict['delta'], dt_discrete)
S0 = best_draw_dict['S0']
I0 = best_draw_dict['I0']



if with_seasonality:
    dbeta = best_draw_dict['dbeta']
else:
    dbeta = 0.0

# Convert all rates to 1/year:
beta0 *= 52
gamma *= 52
mu *= 52
delta *= 52

In [ ]:
print(f"beta0 :\t\t {beta0} -- mean transmission rate in units of 1/year")
print(f"dbeta :\t\t {dbeta} -- amplitude of sinusoidal transmission rate fluctuations")
print(f"betaphase :\t {betaphase} -- phase of sinusoidal transmission rate")
print(f"gamma :\t\t {gamma} -- Recovery rate in units of 1/year")
print(f"mu :\t\t {mu} -- Birth/death rate in units of 1/year")
print(f"delta :\t\t {delta} -- Immunity waning rate in units of 1/year. Equals 1/(duration of immunity)")
print(f"S0 :\t\t {S0} -- initial fraction susceptible")
print(f"I0 :\t\t {I0} -- initial fraction infected")

In [ ]:
import numpy as np
import numpy.linalg as LA

@njit
def get_noise_draws(noise_level, T, dt):
    alpha = dt / noise_level**2
    beta = noise_level**2
    k_shape = alpha
    theta_scale = beta
    noise_draws = np.empty(T)
    for i in range(T):
        DeltaGamma_loc = np.random.gamma(k_shape, theta_scale)
        noise_draws[i] = DeltaGamma_loc / dt
    return noise_draws

def sirs_ode(t, S, I, R, beta0, dbeta, betaphase, gamma, mu, delta, noise_draw):
    # seasonal transmission rate
    beta_t = beta0 * (1 + dbeta*np.sin(2*np.pi*t + betaphase)) * noise_draw
    dSdt = mu - beta_t*S*I - mu*S + delta*R
    dIdt = beta_t*S*I - (mu + gamma)*I
    dRdt = gamma*I - (mu + delta)*R
    return np.array([dSdt, dIdt, dRdt])

def sirs_jacobian(t, S, I, R, beta0, dbeta, betaphase, gamma, mu, delta, noise_draw):
    beta_t = beta0 * (1 + dbeta*np.sin(2*np.pi*t + betaphase)) * noise_draw
    
    # build Jacobian
    J = np.zeros((3,3))
    # partial derivatives of dS/dt (wrt S, I and R)
    J[0,0] = -beta_t*I - mu
    J[0,1] = -beta_t*S
    J[0,2] = delta
    # partial derivatives of dI/dt
    J[1,0] = beta_t*I
    J[1,1] = beta_t*S - (mu + gamma)
    J[1,2] = 0.0
    # partial derivatives of dR/dt
    J[2,0] = 0.0
    J[2,1] = gamma
    J[2,2] = -(mu + delta)
    
    return J

def lyapunov_exponents_sirs(beta0, dbeta, betaphase, gamma, mu, delta, 
                            S0, I0, noise_draws, dt=0.05, burn_in_time=100, total_time=20, t0=0.0):
    
    # number of steps
    N = int(total_time / dt)
    
    S, I, R = S0, I0, 1 - S0 - I0
    
    # initial tangent vectors
    Q = np.eye(3)

    S_timeseries = np.full(N+1, np.nan)
    I_timeseries = np.full(N+1, np.nan)
    R_timeseries = np.full(N+1, np.nan)
    ts = np.full(N+1, np.nan)
    ts[0] = t0
    S_timeseries[0] = S
    I_timeseries[0] = I
    R_timeseries[0] = R
    
    
    t = t0
    lyaps = np.zeros((3, N+1))
    for k in range(1, N+1):
        noise_draw = noise_draws[k]
        # compute the ODE derivatives
        f = sirs_ode(t, S, I, R, beta0, dbeta, betaphase, gamma, mu, delta, noise_draw)
        
        S += dt * f[0]
        I += dt * f[1]
        R += dt * f[2]
        
        if t - t0 >= burn_in_time:
            # compute Jacobian at current step
            J = sirs_jacobian(t, S, I, R, beta0, dbeta, betaphase, gamma, mu, delta, noise_draw)
            # tangent vector update: multiply each column of Q by [I + dt*J]
            M = np.eye(3) + dt*J
            Q = M @ Q  
            
            # do QR decomposition
            Q, Rmat = np.linalg.qr(Q) 
            # get log stretch factors from diagonal of Rmat
            diag = np.abs(np.diag(Rmat))
            lyaps[:,k] = np.log(diag)/dt
        
        t += dt
        ts[k] = t
        S_timeseries[k] = S
        I_timeseries[k] = I
        R_timeseries[k] = R
    
    lyaps[:,0]=lyaps[:,1]
    
    return lyaps, ts, S_timeseries, I_timeseries, R_timeseries

dt = 0.002
#dt = 0.05
#noise_level = 1e-18
noise_level = 0.12/np.sqrt(52)

burn_in_years = 200

time_tot_years = burn_in_years+50

noise_draws = get_noise_draws(noise_level, int(time_tot_years/dt)+1, dt)

LLEs, ts, Ss, Is, Rs = lyapunov_exponents_sirs(beta0, dbeta, betaphase, gamma, mu, delta, 
                               S0, I0, noise_draws, dt=dt, total_time=time_tot_years, burn_in_time=burn_in_years)
print("Estimated Lyapunov exponents:", LLEs)
print("Max mean LE =", np.max(np.mean(LLEs,axis=1)))
print("Max point LLE =", np.max(LLEs))

In [ ]:
plt.figure(figsize=(4.2, 3), dpi=400)
plt.plot(ts, Is, label=r"$I(t)$", color=col1)
plt.legend()
plt.xlim([burn_in_years-0.2, time_tot_years])

In [ ]:
plt.figure(figsize=(4.2, 3), dpi=400)

import matplotlib.cm as cm
col1 = plt.cm.inferno(0.3)
col2 = plt.cm.inferno(0.55)
col3 = plt.cm.inferno(0.8)

nth = 100

ts_minus_burn_in = ts-burn_in_years

#plt.plot(durations_yr, lya_t[0,:], label=r"$\lambda_S$")
plt.plot(ts_minus_burn_in[::nth], LLEs[0,::nth], label=r"$\lambda_1$", color=col1)
plt.plot(ts_minus_burn_in[::nth], LLEs[1,::nth], label=r"$\lambda_2$", color=col2)
plt.plot(ts_minus_burn_in[::nth], LLEs[2,::nth], label=r"$\lambda_3$", color=col3)
#plt.plot(durations_yr, lya_t[1,:], label=r"$\lambda_I$")
#plt.plot(durations_yr, lya_t[2,:], label=r"$\lambda_R$")
plt.axhline(0, ls="--", color="black")
plt.xlim([-0.2, time_tot_years-burn_in_years])
plt.legend(loc="upper right", ncol=3)
#plt.xlabel("Years averaged over")
plt.ylabel("Local Lyapunov exponent (LLE)")
plt.xlabel("Time (years)")
#plt.title("Instantaneous/local Lyapunov exponent")
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)


In [ ]:
years_to_expand = burn_in_years + 2
print(f"Expansion over the first {years_to_expand-burn_in_years} years:")
print(np.prod(np.exp(LLEs[1, ts <= years_to_expand] * dt)))

In [ ]:
years_to_expand = burn_in_years + 10
print(f"Expansion over the first {years_to_expand-burn_in_years} years:")
print(np.prod(np.exp(LLEs[1, ts <= years_to_expand] * dt)))

In [ ]:
lya_t = np.zeros((3, len(ts)))
for i in range(lya_t.shape[0]):
    cs = np.cumsum(LLEs[i, :])
    lya_t[i, :] = cs / np.arange(1, len(ts)+1)

In [ ]:
plt.figure(figsize=(4.2, 3), dpi=400)
for i in range(lya_t.shape[0]):
    cols=[col1, col2, col3]
    to_plot = lya_t[i,:]
    plt.plot(ts-burn_in_years, to_plot, label=rf"$\widebar\lambda_{i+1}$", color=cols[i], alpha=1)
plt.axhline(0)
plt.xlim([-0.2, time_tot_years-burn_in_years])

ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.xlabel("Years averaged over")
plt.ylabel("Cumulative mean LLE")
plt.legend(loc="upper right", ncol=3)